In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

In [ ]:
# self-desighed block

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20,256)
        self.out = nn.Linear(256,10)

    def forward(self,X):
        return self.out(F.relu(self.hidden(X)))


In [ ]:
# sequential block
class Mysequetial(nn.Module):
    def __init__(self, *args) -> None:
        super().__init__()
        for idx, module in enumerate(args):
            # self._modules is a dic
            self._modules[str(idx)] = module

    def forward(self,X):
        # .values()!!!
        for block in self._modules.values():
            X = block(X)
        return X
    

In [ ]:
# incorporate constant parameter and control flow

class FixedHiddenMLP(nn.Module):
    def __init__(self, *args) -> None:
        super().__init__()
        self.const_weight = torch.rand((20,20),requires_grad=False)
        self.linear = nn.Linear(20,20)

    def forward(self,X):
        X = self.linear(X)
        # constant parameters like const_weight and bias=1
        X = F.relu(torch.mm(X,self.const_weight)+1)
        # lazy~
        X = self.linear(X)
        while X.abs().sum() > 1:
            X = X/2
        return X.sum()

In [ ]:
# nested block

class NestMLP(nn.Module):
    def __init__(self, *args) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self,X):
        return self.linear(self.net(X))
    
chimera = nn.Sequential(NestMLP(),nn.Linear(16,20),FixedHiddenMLP())

In [ ]:
# parallel block

class ParallelBlock(nn.Module):
    # need to be passed to self._module
    # allow for transferability
    def __init__(self,net1,net2) -> None:
        super().__init__()

        self.net1 = net1
        self.net2 = net2

    def forward(self,X):
        out1 = self.net1(X)
        out2 = self.net2(X)

        out = torch.cat([out1,out2],dim = 1)
        return out
    
def make_many_blocks(block_class,num_instances,**kwargs):
    layers = []
    for _ in range(num_instances):
        layers.append(block_class(**kwargs))
    return nn.Sequential(*layers)    